# Phase 3 — Probe μ_HR : direction correcte ?

**Question** : Le signal μ_HR produit par Stage 1 a-t-il la bonne *direction* pour améliorer les métriques,
ou est-ce que la magnitude/le mécanisme est le seul problème ?

**Protocole** (zéro réentraînement) :
1. Charger Stage 1 + Stage 2 entraînés (9-node seed42)
2. Extraire `delta_pred = pred_ref - baseline - mu_HR` (UNet pur)
3. Reconstruire avec 5 variantes de μ_HR
4. Comparer F1@p99 → verdict sur l'interface AdaLN

**Variantes testées** :
- V0 `ablation` : `baseline + 0·μ_HR + delta`
- V1 `ref_1x`   : `baseline + 1·μ_HR + delta` (comportement actuel)
- V2 `scale_2x` : `baseline + 2·μ_HR + delta`
- V3 `scale_5x` : `baseline + 5·μ_HR + delta`
- V4 `oracle_α` : `baseline + α*·μ_HR + delta` (α* optimal per-sample L2)

**Décision** :
- `F1@p99(oracle_α) > 0.512` → direction correcte → AdaLN warm-start viable
- `F1@p99(scale_5x) > F1@p99(ref_1x)` → magnitude issue → AdaLN
- `F1@p99(oracle_α) ≤ 0.512` → direction fausse → problème plus profond
- `F1@p99(ablation) > F1@p99(ref_1x)` → μ_HR nuit → investiguer biais Stage 1

In [ ]:
# === Cell 1 : Bootstrap ===
import os, sys, shlex, subprocess
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE_ROOT = Path('/content/drive/MyDrive/climate_data')
    REPO_DIR   = Path('/content/climate_data')
    GIT_URL    = 'https://github.com/leonelkenfack/stcdgm.git'
    GIT_BRANCH = 'four-node-causal'
    if not (REPO_DIR / '.git').exists():
        subprocess.run(shlex.split(f'git clone --depth 200 -b {GIT_BRANCH} {GIT_URL} {REPO_DIR}'), check=True)
    else:
        subprocess.run(shlex.split(f'git -C {REPO_DIR} fetch --depth=200 origin {GIT_BRANCH}'), check=True)
        subprocess.run(shlex.split(f'git -C {REPO_DIR} reset --hard origin/{GIT_BRANCH}'), check=True)
    os.chdir(str(REPO_DIR))
    sys.path.insert(0, str(REPO_DIR / 'src'))
    try:
        import torch_geometric; import cftime; import h5netcdf
        import xbatcher; import diffusers; from omegaconf import OmegaConf
    except ImportError as _e:
        print(f'Installing deps : {_e}')
        _deps = [
            'torch_geometric', 'omegaconf==2.3.0', 'hydra-core==1.3.2',
            'diffusers==0.36.0', 'einops', 'scipy', 'h5py', 'netCDF4',
            'xarray', 'dask', 'zarr', 'safetensors==0.7.0',
            'xbatcher', 'webdataset', 'cftime', 'h5netcdf',
        ]
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + _deps, check=True)
        print('Deps OK.')
else:
    DRIVE_ROOT = Path('c:/Users/reall/Desktop/climate_data')
    REPO_DIR   = DRIVE_ROOT
    sys.path.insert(0, str(REPO_DIR / 'src'))

print(f'DRIVE_ROOT : {DRIVE_ROOT}')
print(f'sys.path[0]: {sys.path[0]}')

In [ ]:
# === Cell 2 : Imports + paths + constantes ===
import json, time, math
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED       = 42
N_STEPS    = 18      # EDM Heun steps — identique à l'éval officielle
K_SAMPLES  = 1       # samples par batch (rapide, probe seulement)
BATCH_SIZE = 4
CFG_SCALE  = 1.0
torch.manual_seed(SEED); np.random.seed(SEED)

# Checkpoints — 9-node en priorité, 6-node en fallback
CKPT_9N = DRIVE_ROOT / 'oracle_9node' / 'seed_42' / 'epoch_last.pth'
CKPT_6N = DRIVE_ROOT / 'oracle_full'  / 'seed_42' / 'epoch_last.pth'
CKPT    = CKPT_9N if CKPT_9N.exists() else CKPT_6N
N_NODES = 9 if CKPT_9N.exists() else 6
print(f'Device  : {DEVICE}')
print(f'Ckpt    : {CKPT}  (N_NODES={N_NODES})')
print(f'Ckpt OK : {CKPT.exists()}')

# Références métriques
REF = {
    'current_9n' : {'rmse': 0.14135, 'pearson': 0.7667, 'f1_p99': 0.4531},
    'noncausal_v4': {'rmse': 0.1243,  'pearson': 0.834,  'f1_p99': 0.512 },
}

# Output
OUT_DIR = DRIVE_ROOT / 'oracle_9node' / 'seed_42' / 'phase3_probe'
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Output  : {OUT_DIR}')

In [ ]:
# === Cell 3 : Charger CONFIG depuis les YAML + artefacts cachés ===
#
# CONFIG n'est pas dans le package — il se charge depuis config/*.yaml
# après os.chdir(REPO_DIR) (fait en Cell 1).
#
# FAST PATH (recommandé) : charger pred_ref, mu_HR, baseline, HR_true
# depuis eval_samples.npz déjà produit par BS43 de st_cdgm_seed42_eval.
# Zéro forward pass nécessaire — tous les tests sont du tensor arithmetic.

import torch
import numpy as np
from omegaconf import OmegaConf

# --- CONFIG ---
CONFIG = OmegaConf.load("config/training_config.yaml")
_corrdiff = OmegaConf.load("config/training_config_corrdiff_normal.yaml")
CONFIG = OmegaConf.merge(CONFIG, _corrdiff)
print(f'CONFIG chargé : data.seq_len={CONFIG.data.seq_len}  '
      f'data.stride={CONFIG.data.stride}')

# --- Chercher eval_samples.npz (plusieurs chemins possibles) ---
_NPZ_CANDIDATES = [
    DRIVE_ROOT / 'oracle_9node' / 'seed_42' / 'eval_samples.npz',
    DRIVE_ROOT / 'oracle_9node' / 'seed_42' / 'final_val' / 'eval_samples.npz',
    DRIVE_ROOT / 'oracle_full'  / 'seed_42' / 'eval_samples.npz',
    DRIVE_ROOT / 'oracle_full'  / 'seed_42' / 'final_val' / 'eval_samples.npz',
]

NPZ_PATH = None
for _p in _NPZ_CANDIDATES:
    if _p.exists():
        NPZ_PATH = _p
        print(f'eval_samples.npz trouvé : {NPZ_PATH}')
        break

if NPZ_PATH is None:
    print('⚠ eval_samples.npz introuvable dans les chemins standards.')
    print('  Chemins testés :')
    for _p in _NPZ_CANDIDATES:
        print(f'    {_p}')
    print()
    print('→ Option A : localiser le fichier sur le Drive et mettre le chemin dans NPZ_PATH')
    print('→ Option B : exécuter Cell 4 (slow path : reconstruit les modèles depuis checkpoint)')
    NPZ_PATH = None
else:
    _npz = np.load(NPZ_PATH, allow_pickle=True)
    print(f'Clés disponibles : {list(_npz.keys())}')


In [ ]:
# === Cell 4 : Charger tenseurs depuis eval_samples.npz (fast path) ===
#
# Le fichier eval_samples.npz contient les artefacts BS43 de l'éval officielle :
#   pred_ref  : [N, 1, H, W] — prédiction Stage 2 complète (baseline + mu_HR + delta)
#   mu_HR     : [N, 1, H, W] — sortie Stage 1 (μ_HR)
#   baseline  : [N, 1, H, W] — baseline log-précip
#   HR_true   : [N, 1, H, W] — vérité terrain (test split K9)
#   valid_mask : [N, 1, H, W] — masque terre/mer (optionnel)

assert NPZ_PATH is not None, (
    "eval_samples.npz non trouvé. Voir Cell 3 pour les options A/B."
)

_npz = np.load(NPZ_PATH, allow_pickle=True)
print('Clés dans eval_samples.npz :', list(_npz.keys()))

# Charger et convertir en tenseurs float32
def _load(key, fallback=None):
    if key in _npz:
        return torch.from_numpy(_npz[key].astype(np.float32))
    if fallback is not None:
        print(f'  ⚠ clé "{key}" absente — utilise fallback')
        return fallback
    raise KeyError(f'Clé "{key}" absente de eval_samples.npz')

# Les noms de clés peuvent varier légèrement selon la version du notebook
pred_ref  = _load('pred_ref',  _load('predictions', None))
mu_HR     = _load('mu_HR',     _load('mu_hr', None))
baseline  = _load('baseline',  _load('baseline_log', None))
HR_true   = _load('HR_true',   _load('hr_true', _load('targets', None)))
masks     = _load('valid_mask', torch.ones_like(HR_true))

# Garantir shape [N, 1, H, W]
for name, t in [('pred_ref', pred_ref), ('mu_HR', mu_HR),
                 ('baseline', baseline), ('HR_true', HR_true)]:
    if t.dim() == 3:
        t = t.unsqueeze(1)
    print(f'  {name:12s} shape={tuple(t.shape)}  '
          f'mean={t.mean():.4f}  std={t.std():.4f}')

# Extraire delta UNet pur
# HR_pred = baseline + mu_HR + delta  =>  delta = pred_ref - baseline - mu_HR
delta_pred = pred_ref - baseline - mu_HR

print(f'\ndelta_pred  shape={tuple(delta_pred.shape)}  '
      f'mean={delta_pred.mean():.5f}  std={delta_pred.std():.5f}')
print(f'mu_HR norme moyenne par sample : '
      f'{mu_HR.view(mu_HR.shape[0],-1).norm(dim=1).mean():.4f}')
print(f'delta norme moyenne par sample : '
      f'{delta_pred.view(delta_pred.shape[0],-1).norm(dim=1).mean():.4f}')


In [ ]:
# === Cell 5 : Construire les 5 variantes — pur tensor arithmetic, zéro forward pass ===

N = HR_true.shape[0]

# Oracle α* per-sample (L2-optimal)
# α* = <μ_HR, HR_true - baseline - delta> / ‖μ_HR‖²
residual_true = HR_true - baseline - delta_pred        # ce que μ_HR devrait expliquer
num   = (mu_HR * residual_true).view(N, -1).sum(dim=1)
denom = mu_HR.view(N, -1).pow(2).sum(dim=1).clamp_min(1e-8)
alpha_opt = (num / denom).clamp(0.0, 10.0)            # [N]

print(f'α* stats : mean={alpha_opt.mean():.3f}  std={alpha_opt.std():.3f}  '
      f'min={alpha_opt.min():.3f}  max={alpha_opt.max():.3f}')
print(f'  α* < 0.5  : {(alpha_opt < 0.5).float().mean()*100:.1f}%  (μ_HR sur-pondéré)')
print(f'  0.5≤α*<1.5: {((alpha_opt >= 0.5) & (alpha_opt < 1.5)).float().mean()*100:.1f}%  (magnitude OK)')
print(f'  α* ≥ 1.5  : {(alpha_opt >= 1.5).float().mean()*100:.1f}%  (μ_HR sous-pondéré)')

# Cinq variantes de reconstruction
preds = {
    'ablation'    : baseline + delta_pred,
    'ref_1x'      : baseline + mu_HR                              + delta_pred,
    'scale_2x'    : baseline + 2.0 * mu_HR                        + delta_pred,
    'scale_5x'    : baseline + 5.0 * mu_HR                        + delta_pred,
    'oracle_alpha': baseline + alpha_opt.view(N,1,1,1) * mu_HR    + delta_pred,
}

# Sanity : ref_1x doit être ≈ pred_ref
_diff = (preds['ref_1x'] - pred_ref).abs().mean().item()
print(f'\nSanity ref_1x ≈ pred_ref : diff_mean={_diff:.2e}  '
      f'{"✓ OK" if _diff < 1e-4 else "⚠ écart non négligeable"}')


In [ ]:
# === Cell 6 : Métriques pour chaque variante ===

def _pearson(a, b, eps=1e-8):
    a_c = a - a.mean(); b_c = b - b.mean()
    return (a_c * b_c).sum().item() / (a_c.norm().item() * b_c.norm().item() + eps)

def _f1_extremes(pred_np, true_np, pct=99.0):
    """F1 pour les extrêmes au-delà du percentile `pct`."""
    thresh = float(np.percentile(true_np, pct))
    pred_ext = (pred_np >= thresh)
    true_ext = (true_np >= thresh)
    tp = float((pred_ext & true_ext).sum())
    fp = float((pred_ext & ~true_ext).sum())
    fn = float((~pred_ext & true_ext).sum())
    prec = tp / (tp + fp + 1e-8)
    rec  = tp / (tp + fn + 1e-8)
    return 2 * prec * rec / (prec + rec + 1e-8)

def compute_metrics(pred, true, mask):
    p = pred * mask; t = true * mask
    n = mask.sum().clamp_min(1.0)
    rmse    = ((p - t).pow(2).sum() / n).sqrt().item()
    mae     = (p - t).abs().sum().item() / n.item()
    pearson = _pearson(p.flatten(), t.flatten())
    # F1 sur le domaine valide seulement
    _pv = pred[mask.bool()].numpy(); _tv = true[mask.bool()].numpy()
    f1_p95 = _f1_extremes(_pv, _tv, 95.0)
    f1_p99 = _f1_extremes(_pv, _tv, 99.0)
    return {'rmse': rmse, 'mae': mae, 'pearson': pearson,
            'f1_p95': f1_p95, 'f1_p99': f1_p99}

print('=== Métriques par variante ===')
print(f'{"Variant":15s}  {"RMSE":>8}  {"Pearson":>8}  {"F1@p95":>8}  {"F1@p99":>8}')
print('-' * 60)
results = {}
for name, pred in preds.items():
    m = compute_metrics(pred, HR_true, masks)
    results[name] = m
    marker = ' ← actuel' if name == 'ref_1x' else ''
    print(f'{name:15s}  {m["rmse"]:8.5f}  {m["pearson"]:8.4f}  '
          f'{m["f1_p95"]:8.4f}  {m["f1_p99"]:8.4f}{marker}')
print('-' * 60)
print(f'{"noncausal_v4":15s}  {0.12430:8.5f}  {0.83440:8.4f}  {"   —":>8}  {0.51200:8.4f}  ← TARGET')
print()
print(f'α* moyen = {alpha_opt.mean():.3f}  '
      f'(1.0 = magnitude actuelle correcte, >1 = μ_HR sous-pondéré)')


In [ ]:
# === Cell 7 : Décision automatique + save ===

f1_ref     = results['ref_1x']['f1_p99']
f1_abl     = results['ablation']['f1_p99']
f1_2x      = results['scale_2x']['f1_p99']
f1_5x      = results['scale_5x']['f1_p99']
f1_oracle  = results['oracle_alpha']['f1_p99']
F1_TARGET  = 0.512   # noncausal v4
alpha_mean = float(alpha_opts.mean().item())
alpha_std  = float(alpha_opts.std().item())

print('=' * 70)
print('VERDICT PHASE 3 — PROBE μ_HR')
print('=' * 70)
print(f'  F1@p99 ablation (0×) : {f1_abl:.4f}')
print(f'  F1@p99 ref_1x        : {f1_ref:.4f}   (actuel)')
print(f'  F1@p99 scale_2x      : {f1_2x:.4f}')
print(f'  F1@p99 scale_5x      : {f1_5x:.4f}')
print(f'  F1@p99 oracle_α      : {f1_oracle:.4f}')
print(f'  F1@p99 noncausal v4  : {F1_TARGET:.4f}   (target)')
print(f'  α* moyen             : {alpha_mean:.3f} ± {alpha_std:.3f}')
print()

# Critères de décision
oracle_beats_noncausal = f1_oracle > F1_TARGET
scaling_helps          = f1_5x > f1_ref + 0.005
mu_HR_hurts            = f1_abl > f1_ref + 0.005
direction_correct      = alpha_mean > 0.5   # α* positif et raisonnable

if mu_HR_hurts:
    verdict = 'VERDICT_D_MU_HR_HARMFUL'
    print('  ✗ VERDICT D : μ_HR NUIT au modèle')
    print('  ▶ F1@p99(ablation) > F1@p99(ref) — retirer μ_HR de la reconstruction améliore')
    print('  ▶ Stage 1 produit un μ_HR biaisé. Investiguer biais de normalisation.')
    print('  ▶ NE PAS faire AdaLN — corriger Stage 1 ou supprimer μ_HR de la reconstruction.')
elif oracle_beats_noncausal and direction_correct:
    verdict = 'VERDICT_A_DIRECTION_OK_ADALN_GO'
    print('  ✓ VERDICT A : direction μ_HR CORRECTE — AdaLN warm-start VIABLE')
    print(f'  ▶ F1@p99(oracle_α={alpha_mean:.2f}×) = {f1_oracle:.4f} > {F1_TARGET:.4f} (noncausal)')
    print('  ▶ Le signal causal est dans la bonne direction mais mal calibré en magnitude.')
    print('  ▶ AdaLN apprendrait la bonne pondération dynamique de μ_HR.')
    print('  ▶ NEXT : warm-start AdaLN sur Stage 2 existant (~10-15h).')
elif scaling_helps:
    verdict = 'VERDICT_B_SCALING_HELPS_ADALN_GO'
    print('  ◑ VERDICT B : le scaling aide — AdaLN warm-start RECOMMANDÉ')
    print(f'  ▶ F1@p99(5×) = {f1_5x:.4f} > F1@p99(1×) = {f1_ref:.4f}')
    print('  ▶ μ_HR est sous-pondéré. AdaLN apprendrait l\'échelle optimale.')
    print('  ▶ NEXT : warm-start AdaLN (~10-15h).')
else:
    verdict = 'VERDICT_C_DIRECTION_WRONG'
    print('  ✗ VERDICT C : direction μ_HR INCORRECTE ou insuffisante')
    print(f'  ▶ F1@p99(oracle_α) = {f1_oracle:.4f} ≤ {F1_TARGET:.4f} même avec α* optimal')
    print('  ▶ μ_HR n\'a pas la bonne direction pour améliorer F1@p99.')
    print('  ▶ Problème plus profond : soit Stage 1 est biaisé, soit l\'information')
    print('    causale n\'est pas dans les variables du DAG.')
    print('  ▶ Investiguer : biais Stage 1, variables manquantes, ou revoir architecture.')

# Save
save = {
    'verdict': verdict,
    'n_nodes': N_NODES,
    'seed': SEED,
    'alpha_opt': {'mean': alpha_mean, 'std': alpha_std,
                  'min': float(alpha_opts.min()), 'max': float(alpha_opts.max())},
    'f1_p99': {
        'ablation': f1_abl, 'ref_1x': f1_ref,
        'scale_2x': f1_2x, 'scale_5x': f1_5x, 'oracle_alpha': f1_oracle,
        'noncausal_v4': F1_TARGET,
    },
    'full_metrics': results,
    'decision_flags': {
        'oracle_beats_noncausal': oracle_beats_noncausal,
        'scaling_helps': scaling_helps,
        'mu_HR_hurts': mu_HR_hurts,
        'direction_correct': direction_correct,
    },
}
save_path = OUT_DIR / 'probe_metrics.json'
with open(save_path, 'w') as f:
    json.dump(save, f, indent=2)
print(f'\nSauvegardé : {save_path}')

In [ ]:
# === Cell 8 : Visualisations ===
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# --- F1@p99 bar chart ---
names  = ['ablation\n(0×)', 'ref_1×\n(actuel)', 'scale_2×', 'scale_5×', 'oracle_α*']
f1vals = [f1_abl, f1_ref, f1_2x, f1_5x, f1_oracle]
colors = ['#aaa', '#e74c3c', '#3498db', '#2980b9', '#27ae60']
bars = axes[0].bar(names, f1vals, color=colors, edgecolor='white', width=0.6)
axes[0].axhline(F1_TARGET,  ls='--', color='orange', lw=1.8, label=f'noncausal v4 = {F1_TARGET}')
axes[0].axhline(f1_ref,     ls=':',  color='red',    lw=1.2, label=f'actuel = {f1_ref:.4f}')
for bar, val in zip(bars, f1vals):
    axes[0].text(bar.get_x() + bar.get_width()/2, val + 0.003,
                 f'{val:.4f}', ha='center', va='bottom', fontsize=8)
axes[0].set_ylabel('F1@p99')
axes[0].set_title('F1@p99 par variante μ_HR')
axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(0, max(f1vals + [F1_TARGET]) * 1.15)

# --- Distribution alpha_opt ---
axes[1].hist(alpha_opts.numpy(), bins=40, color='#27ae60', edgecolor='white', alpha=0.85)
axes[1].axvline(alpha_mean, color='black', ls='--', lw=1.5, label=f'mean={alpha_mean:.2f}')
axes[1].axvline(1.0, color='red', ls=':', lw=1.2, label='α=1 (current)')
axes[1].set_xlabel('α* optimal per-sample')
axes[1].set_ylabel('Count')
axes[1].set_title('Distribution α* oracle\n(1.0 = magnitude actuelle)')
axes[1].legend(fontsize=9); axes[1].grid(True, alpha=0.3)

# --- RMSE bar chart ---
rmse_names = list(results.keys()) + ['noncausal_v4']
rmse_vals  = [results[k]['rmse'] for k in results] + [0.1243]
clrs2 = colors + ['orange']
axes[2].bar(rmse_names, rmse_vals, color=clrs2, edgecolor='white', width=0.6)
axes[2].set_xticklabels(['abl', 'ref_1×', '2×', '5×', 'oracle', 'noncausal'],
                         rotation=30, ha='right', fontsize=9)
axes[2].set_ylabel('RMSE')
axes[2].set_title('RMSE par variante')
axes[2].grid(True, alpha=0.3)

plt.suptitle(f'Phase 3 Probe μ_HR — {verdict}', fontsize=11, y=1.01)
plt.tight_layout()
fig_path = OUT_DIR / 'probe_viz.png'
plt.savefig(fig_path, dpi=130, bbox_inches='tight')
plt.show()
print(f'Figure : {fig_path}')